In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv('mumbai_weather.csv')

print(df.head())

# Fill missing average temperature
df['tavg'] = df['tavg'].fillna((df['tmin'] + df['tmax']) / 2)

# Extract Year
df['Year'] = df['time'].apply(lambda x: x.split('-')[2])

# Fill remaining missing values
df['tavg'] = df.groupby('Year')['tavg'].transform(lambda x: x.fillna(x.mean()))
df['tmin'] = df['tmin'].fillna(df['tavg'] - 5)
df['tmax'] = df['tmax'].fillna(df['tavg'] + 5)
df['prcp'] = df['prcp'].fillna(0)

print('\nNull Values:\n')
print(df.isnull().sum())

print('\nDataset Shape:', df.shape)

# ---------------- MAP FUNCTION ----------------
def mapper(df):
    result = []

    for _, row in df.iterrows():
        year = row['Year']
        tavg = row['tavg']
        result.append((year, tavg))

    return result

mapped_output = mapper(df)

# ---------------- REDUCE FUNCTION ----------------
def reducer(mapped_output):
    grouped = {}

    for year, tavg in mapped_output:
        if year in grouped:
            grouped[year].append(tavg)
        else:
            grouped[year] = [tavg]

    avg_by_year = {
        year: sum(tavgs) / len(tavgs)
        for year, tavgs in grouped.items()
    }

    return avg_by_year

reducer_output = reducer(mapped_output)

print('\nAverage Temperature By Year:\n')
print(reducer_output)

# ---------------- HOTTEST & COLDEST YEAR ----------------
hottest_year = max(reducer_output, key=reducer_output.get)
coldest_year = min(reducer_output, key=reducer_output.get)

print('\nHottest year:', hottest_year,
      'with avg temp =', reducer_output[hottest_year])

print('Coldest year:', coldest_year,
      'with avg temp =', reducer_output[coldest_year])

# =========================================================
# GRAPH 1 : Average Temperature By Year
# =========================================================

years = list(reducer_output.keys())
temps = list(reducer_output.values())

plt.figure(figsize=(10, 5))
plt.plot(years, temps, marker='o')
plt.title('Average Temperature By Year')
plt.xlabel('Year')
plt.ylabel('Average Temperature')
plt.grid(True)
plt.show()

# =========================================================
# GRAPH 2 : Hottest vs Coldest Year
# =========================================================

plt.figure(figsize=(6, 5))

plt.bar(
    [hottest_year, coldest_year],
    [reducer_output[hottest_year], reducer_output[coldest_year]]
)

plt.title('Hottest vs Coldest Year')
plt.xlabel('Year')
plt.ylabel('Average Temperature')

plt.show()

# =========================================================
# GRAPH 3 : Temperature Distribution
# =========================================================

plt.figure(figsize=(8, 5))
plt.hist(df['tavg'], bins=20)
plt.title('Temperature Distribution')
plt.xlabel('Average Temperature')
plt.ylabel('Frequency')
plt.show()
